In [ ]:
# ScamShield PK - Pakistani Scam Message Detector

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print("ScamShield PK setup successful!")

ScamShield PK setup successful!


In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

urllib.request.urlretrieve(url, "spam.csv")

df = pd.read_csv("spam.csv", sep="\t", header=None,
                 names=["label", "message"])

print("Dataset loaded!")
print("Shape:", df.shape)
df.head()

Dataset loaded!
Shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
print("Message distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDataset information:")
print(df.info())

Message distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [ ]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())


Training messages: 4457
Testing messages: 1115

Training distribution:
label
ham     3859
spam     598
Name: count, dtype: int64


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

print("Training data shape:", X_train_vectors.shape)
print("Testing data shape:", X_test_vectors.shape)
print("Text successfully converted into numbers!")

Training data shape: (4457, 7403)
Testing data shape: (1115, 7403)
Text successfully converted into numbers!


In [ ]:
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train_vectors, y_train)

print("ScamShield PK model trained successfully!")

# Make predictions
y_pred = model.predict(X_test_vectors)

print("Predictions completed!")

ScamShield PK model trained successfully!
Predictions completed!


In [ ]:
# Evaluate ScamShield PK

print("MODEL ACCURACY:")
print(accuracy_score(y_test, y_pred))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))

MODEL ACCURACY:
0.967713004484305

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115


CONFUSION MATRIX:
[[966   0]
 [ 36 113]]


In [ ]:
# Simple baseline: predict every message as ham

baseline_predictions = ["ham"] * len(y_test)

print("BASELINE ACCURACY:")
print(accuracy_score(y_test, baseline_predictions))

print("\nBASELINE CLASSIFICATION REPORT:")
print(classification_report(
    y_test,
    baseline_predictions,
    zero_division=0
))


BASELINE ACCURACY:
0.8663677130044843

BASELINE CLASSIFICATION REPORT:
              precision    recall  f1-score   support

         ham       0.87      1.00      0.93       966
        spam       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115



In [ ]:
# Find spam messages our model missed

results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "message": X_test.values
})

missed_spam = results[
    (results["actual"] == "spam") &
    (results["predicted"] == "ham")
]

print("Missed spam messages:", len(missed_spam))
print("\nFive examples:")

for message in missed_spam["message"].head(5):
    print("-", message)

Missed spam messages: 36

Five examples:
- You are now unsubscribed all services. Get tons of sexy babes or hunks straight to your phone! go to http://gotbabes.co.uk. No subscriptions.
- FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
- Dear Voucher Holder 2 claim your 1st class airport lounge passes when using Your holiday voucher call 08704439680. When booking quote 1st class x 2
- Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort time 09099726429 JANINExx Calls£1/minMobsmoreLKPOBOX177HP51FL
- ringtoneking 84484


In [ ]:
pakistani_messages = [
    "Your bank account is blocked. Verify your details immediately.",
    "Congratulations! You have won Rs. 50000. Claim your prize now.",
    "Your Easypaisa account requires verification. Click the link.",
    "JazzCash: Your payment is pending. Confirm your PIN.",
    "Your parcel is held. Pay the delivery fee using this link.",
    "Your OTP is 482913. Do not share it with anyone.",
    "Your electricity bill is due. Pay before the deadline.",
    "Are you coming to school tomorrow?",
    "Please send me the notes from today's class.",
    "Your appointment is confirmed for Monday.",
    "I have transferred the money. Please check.",
    "Call me when you reach home."
]

print("Pakistani test messages:", len(pakistani_messages))

Pakistani test messages: 12


In [ ]:
# Convert Pakistani messages into numerical features
pakistani_vectors = vectorizer.transform(pakistani_messages)

# Get predictions
pakistani_predictions = model.predict(pakistani_vectors)

# Display results
for message, prediction in zip(
    pakistani_messages, pakistani_predictions
):
    print("Message:", message)
    print("AI prediction:", prediction.upper())
    print("-" * 60)

Message: Your bank account is blocked. Verify your details immediately.
AI prediction: HAM
------------------------------------------------------------
Message: Congratulations! You have won Rs. 50000. Claim your prize now.
AI prediction: SPAM
------------------------------------------------------------
Message: Your Easypaisa account requires verification. Click the link.
AI prediction: HAM
------------------------------------------------------------
Message: JazzCash: Your payment is pending. Confirm your PIN.
AI prediction: HAM
------------------------------------------------------------
Message: Your parcel is held. Pay the delivery fee using this link.
AI prediction: HAM
------------------------------------------------------------
Message: Your OTP is 482913. Do not share it with anyone.
AI prediction: HAM
------------------------------------------------------------
Message: Your electricity bill is due. Pay before the deadline.
AI prediction: HAM
---------------------------------

In [ ]:
pakistani_labels = [
    "spam",  # Bank verification link
    "spam",  # Fake prize
    "spam",  # Easypaisa verification link
    "spam",  # JazzCash PIN request
    "spam",  # Parcel payment link
    "ham",   # OTP notification
    "ham",   # Bill reminder (assumed legitimate)
    "ham",   # School conversation
    "ham",   # Class notes
    "ham",   # Appointment
    "ham",   # Transfer confirmation
    "ham"    # Call me
]

print("Messages:", len(pakistani_messages))
print("Labels:", len(pakistani_labels))

Messages: 12
Labels: 12


In [ ]:
local_train_messages = [
    "Your bank account will be suspended. Verify through this link.",
    "Congratulations! You have been selected for a cash reward.",
    "JazzCash payment failed. Send your PIN to restore access.",
    "Your parcel is waiting. Pay a fee through this link.",
    "Claim your free mobile balance by verifying your account.",
    "Urgent: Your wallet account needs immediate verification.",

    "Kal school kitne baje start hoga?",
    "Please send me the homework for tomorrow.",
    "Your payment has been received successfully.",
    "Your electricity bill payment was received. Thank you.",
    "I will call you after reaching home.",
    "Your appointment is confirmed for tomorrow."
]

local_train_labels = [
    "spam", "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham"
]

print("Local training examples:", len(local_train_messages))
print("Labels:", len(local_train_labels))

Local training examples: 12
Labels: 12


In [ ]:
# Combine original training data with Pakistani examples

X_combined = pd.concat([
    X_train.reset_index(drop=True),
    pd.Series(local_train_messages)
], ignore_index=True)

y_combined = pd.concat([
    y_train.reset_index(drop=True),
    pd.Series(local_train_labels)
], ignore_index=True)

# Rebuild TF-IDF using training data only
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_combined_vectors = vectorizer.fit_transform(X_combined)

# Train improved model
model = LogisticRegression(max_iter=1000)
model.fit(X_combined_vectors, y_combined)

print("ScamShield PK retrained successfully!")
print("Total training messages:", len(X_combined))

ScamShield PK retrained successfully!
Total training messages: 4469


In [ ]:
# Convert Pakistani test messages using the updated vectorizer
pakistani_vectors = vectorizer.transform(pakistani_messages)

# Predict using our improved model
improved_predictions = model.predict(pakistani_vectors)

# Display results
for message, prediction in zip(
    pakistani_messages, improved_predictions
):
    print("Message:", message)
    print("Prediction:", prediction.upper())
    print("-" * 50)

Message: Your bank account is blocked. Verify your details immediately.
Prediction: HAM
--------------------------------------------------
Message: Congratulations! You have won Rs. 50000. Claim your prize now.
Prediction: SPAM
--------------------------------------------------
Message: Your Easypaisa account requires verification. Click the link.
Prediction: HAM
--------------------------------------------------
Message: JazzCash: Your payment is pending. Confirm your PIN.
Prediction: HAM
--------------------------------------------------
Message: Your parcel is held. Pay the delivery fee using this link.
Prediction: HAM
--------------------------------------------------
Message: Your OTP is 482913. Do not share it with anyone.
Prediction: HAM
--------------------------------------------------
Message: Your electricity bill is due. Pay before the deadline.
Prediction: HAM
--------------------------------------------------
Message: Are you coming to school tomorrow?
Prediction: HAM
---

In [ ]:
# Examine prediction probabilities

probabilities = model.predict_proba(pakistani_vectors)

for message, prediction, probability in zip(
    pakistani_messages,
    improved_predictions,
    probabilities
):
    spam_probability = probability[
        list(model.classes_).index("spam")
    ]

    print("Message:", message)
    print("Prediction:", prediction.upper())
    print("Spam probability:", round(spam_probability * 100, 2), "%")
    print("-" * 55)

Message: Your bank account is blocked. Verify your details immediately.
Prediction: HAM
Spam probability: 30.8 %
-------------------------------------------------------
Message: Congratulations! You have won Rs. 50000. Claim your prize now.
Prediction: SPAM
Spam probability: 76.14 %
-------------------------------------------------------
Message: Your Easypaisa account requires verification. Click the link.
Prediction: HAM
Spam probability: 28.47 %
-------------------------------------------------------
Message: JazzCash: Your payment is pending. Confirm your PIN.
Prediction: HAM
Spam probability: 11.18 %
-------------------------------------------------------
Message: Your parcel is held. Pay the delivery fee using this link.
Prediction: HAM
Spam probability: 22.28 %
-------------------------------------------------------
Message: Your OTP is 482913. Do not share it with anyone.
Prediction: HAM
Spam probability: 7.12 %
-------------------------------------------------------
Message: Y

In [ ]:
from sklearn.metrics import classification_report

# Recreate test vectors using the updated vectorizer
X_test_updated = vectorizer.transform(X_test)

# Get spam probabilities
test_probabilities = model.predict_proba(X_test_updated)

spam_index = list(model.classes_).index("spam")

spam_scores = test_probabilities[:, spam_index]

# Compare two thresholds
for threshold in [0.5, 0.3]:

    predictions = [
        "spam" if score >= threshold else "ham"
        for score in spam_scores
    ]

    print("\nTHRESHOLD:", threshold)
    print(classification_report(
        y_test,
        predictions,
        zero_division=0
    ))


THRESHOLD: 0.5
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.75      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115


THRESHOLD: 0.3
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.96      0.87      0.92       149

    accuracy                           0.98      1115
   macro avg       0.97      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
# Apply a 30% spam threshold to Pakistani messages

pakistani_probabilities = model.predict_proba(pakistani_vectors)

spam_index = list(model.classes_).index("spam")

for message, probability in zip(
    pakistani_messages,
    pakistani_probabilities
):
    spam_score = probability[spam_index]

    prediction = "SPAM" if spam_score >= 0.30 else "HAM"

    print("Message:", message)
    print("Spam probability:", round(spam_score * 100, 2), "%")
    print("Prediction:", prediction)
    print("-" * 50)

Message: Your bank account is blocked. Verify your details immediately.
Spam probability: 30.8 %
Prediction: SPAM
--------------------------------------------------
Message: Congratulations! You have won Rs. 50000. Claim your prize now.
Spam probability: 76.14 %
Prediction: SPAM
--------------------------------------------------
Message: Your Easypaisa account requires verification. Click the link.
Spam probability: 28.47 %
Prediction: HAM
--------------------------------------------------
Message: JazzCash: Your payment is pending. Confirm your PIN.
Spam probability: 11.18 %
Prediction: HAM
--------------------------------------------------
Message: Your parcel is held. Pay the delivery fee using this link.
Spam probability: 22.28 %
Prediction: HAM
--------------------------------------------------
Message: Your OTP is 482913. Do not share it with anyone.
Spam probability: 7.12 %
Prediction: HAM
--------------------------------------------------
Message: Your electricity bill is due. 

In [ ]:
threshold = 0.25

for message, probability in zip(
    pakistani_messages,
    pakistani_probabilities
):
    spam_score = probability[
        list(model.classes_).index("spam")
    ]

    prediction = (
        "SPAM" if spam_score >= threshold else "HAM"
    )

    print(message)
    print("Spam probability:", round(spam_score * 100, 2), "%")
    print("Prediction:", prediction)
    print("-" * 50)

Your bank account is blocked. Verify your details immediately.
Spam probability: 30.8 %
Prediction: SPAM
--------------------------------------------------
Congratulations! You have won Rs. 50000. Claim your prize now.
Spam probability: 76.14 %
Prediction: SPAM
--------------------------------------------------
Your Easypaisa account requires verification. Click the link.
Spam probability: 28.47 %
Prediction: SPAM
--------------------------------------------------
JazzCash: Your payment is pending. Confirm your PIN.
Spam probability: 11.18 %
Prediction: HAM
--------------------------------------------------
Your parcel is held. Pay the delivery fee using this link.
Spam probability: 22.28 %
Prediction: HAM
--------------------------------------------------
Your OTP is 482913. Do not share it with anyone.
Spam probability: 7.12 %
Prediction: HAM
--------------------------------------------------
Your electricity bill is due. Pay before the deadline.
Spam probability: 7.59 %
Prediction: 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

threshold = 0.25

predictions_25 = [
    "spam" if score >= threshold else "ham"
    for score in spam_scores
]

print("25% THRESHOLD RESULTS")

print(classification_report(
    y_test,
    predictions_25,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions_25))

25% THRESHOLD RESULTS
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.93      0.89      0.91       149

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
[[956  10]
 [ 17 132]]


In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr

# Our chosen experimental threshold
THRESHOLD = 0.25

def detect_scam(message):

    if not message.strip():
        return "Please enter a message.", ""

    # Convert message into numerical features
    message_vector = vectorizer.transform([message])

    # Get spam probability
    probabilities = model.predict_proba(message_vector)[0]

    spam_index = list(model.classes_).index("spam")
    spam_probability = probabilities[spam_index]

    # Make prediction
    if spam_probability >= THRESHOLD:
        result = "⚠️ POTENTIAL SCAM DETECTED"
    else:
        result = "✅ NO SPAM DETECTED"

    confidence = f"Estimated spam probability: {spam_probability * 100:.2f}%"

    return result, confidence


# Design the application
demo = gr.Interface(
    fn=detect_scam,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Paste an SMS message here..."
    ),
    outputs=[
        gr.Textbox(label="ScamShield PK Result"),
        gr.Textbox(label="Analysis")
    ],
    title="🇵🇰 ScamShield PK",
    description="AI-powered SMS scam detection prototype.",
    article="Prototype only — not financial advice. A message marked safe may still be fraudulent."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f576e1587e24ac8f45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import joblib

joblib.dump(model, "scamshield_model.pkl")
joblib.dump(vectorizer, "scamshield_vectorizer.pkl")

print("Model saved!")
print("Vectorizer saved!")

Model saved!
Vectorizer saved!


In [ ]:
from google.colab import files

files.download("scamshield_model.pkl")
files.download("scamshield_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
app_code = '''
import gradio as gr
import joblib

model = joblib.load("scamshield_model.pkl")
vectorizer = joblib.load("scamshield_vectorizer.pkl")

THRESHOLD = 0.25

def detect_scam(message):
    if not message or not message.strip():
        return "Please enter a message.", ""

    vector = vectorizer.transform([message])
    probabilities = model.predict_proba(vector)[0]

    spam_index = list(model.classes_).index("spam")
    spam_probability = probabilities[spam_index]

    if spam_probability >= THRESHOLD:
        result = "⚠️ POTENTIAL SCAM DETECTED"
    else:
        result = "✅ NO SPAM DETECTED"

    analysis = (
        f"Estimated spam probability: "
        f"{spam_probability * 100:.2f}%"
    )

    return result, analysis

demo = gr.Interface(
    fn=detect_scam,
    inputs=gr.Textbox(
        lines=5,
        label="Paste your SMS message"
    ),
    outputs=[
        gr.Textbox(label="ScamShield PK Result"),
        gr.Textbox(label="Analysis")
    ],
    title="🇵🇰 ScamShield PK",
    description="AI-powered SMS scam detection prototype.",
    article="Prototype only — not financial advice. "
            "A message marked safe may still be fraudulent."
)

demo.launch()
'''

with open("app.py", "w") as f:
    f.write(app_code)

with open("requirements.txt", "w") as f:
    f.write("gradio\\nscikit-learn\\njoblib\\n")

print("app.py created!")
print("requirements.txt created!")

app.py created!
requirements.txt created!


In [ ]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sklearn
import gradio
import joblib

print("scikit-learn:", sklearn.__version__)
print("Gradio:", gradio.__version__)
print("Joblib:", joblib.__version__)

scikit-learn: 1.6.1
Gradio: 6.31.0
Joblib: 1.6.0


In [ ]:
with open("requirements.txt", "w") as f:
    f.write(
        "gradio==6.31.0\n"
        "scikit-learn==1.6.1\n"
        "joblib==1.6.0\n"
    )

print("Dependencies locked successfully!")

Dependencies locked successfully!


In [ ]:
def explain_prediction(message, top_n=5):
    message_vector = vectorizer.transform([message])

    feature_names = vectorizer.get_feature_names_out()
    nonzero_indices = message_vector.nonzero()[1]

    contributions = []

    for index in nonzero_indices:
        word = feature_names[index]
        value = (
            message_vector[0, index]
            * model.coef_[0][index]
        )

        if value > 0:
            contributions.append((word, value))

    contributions.sort(key=lambda x: x[1], reverse=True)

    if not contributions:
        return "No strong spam-related word signals were identified."

    explanation = "\n".join(
        [f"• {word}" for word, score in contributions[:top_n]]
    )

    return explanation

In [ ]:
from sklearn.model_selection import train_test_split

X = df["message"]
y = df["label"]

# First split: 70% training, 30% remaining
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide remaining data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 3900
Validation: 836
Testing: 836


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Convert text into numerical features
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_vectors = vectorizer.fit_transform(X_train)
X_val_vectors = vectorizer.transform(X_val)

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectors, y_train)

print("Model training complete!")
print("Training features:", X_train_vectors.shape)
print("Validation features:", X_val_vectors.shape)

Model training complete!
Training features: (3900, 6860)
Validation features: (836, 6860)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Get spam probabilities on validation data
val_probabilities = model.predict_proba(X_val_vectors)

spam_index = list(model.classes_).index("spam")
spam_scores = val_probabilities[:, spam_index]

thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:
    predictions = [
        "spam" if score >= threshold else "ham"
        for score in spam_scores
    ]

    precision = precision_score(
        y_val, predictions, pos_label="spam"
    )
    recall = recall_score(
        y_val, predictions, pos_label="spam"
    )
    f1 = f1_score(
        y_val, predictions, pos_label="spam"
    )

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1-score:  {f1:.3f}")


Threshold: 0.15
Precision: 0.849
Recall:    0.955
F1-score:  0.899

Threshold: 0.2
Precision: 0.929
Recall:    0.929
F1-score:  0.929

Threshold: 0.25
Precision: 0.952
Recall:    0.884
F1-score:  0.917

Threshold: 0.3
Precision: 0.979
Recall:    0.848
F1-score:  0.909

Threshold: 0.35
Precision: 0.979
Recall:    0.830
F1-score:  0.899

Threshold: 0.4
Precision: 0.989
Recall:    0.812
F1-score:  0.892

Threshold: 0.5
Precision: 1.000
Recall:    0.714
F1-score:  0.833


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

THRESHOLD = 0.20

# Transform the untouched test messages
X_test_vectors = vectorizer.transform(X_test)

# Get spam probabilities
test_probabilities = model.predict_proba(X_test_vectors)
spam_index = list(model.classes_).index("spam")
spam_scores = test_probabilities[:, spam_index]

# Apply the selected threshold
y_test_pred = [
    "spam" if score >= THRESHOLD else "ham"
    for score in spam_scores
]

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Accuracy: 0.9700956937799043

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       724
        spam       0.87      0.92      0.89       112

    accuracy                           0.97       836
   macro avg       0.93      0.95      0.94       836
weighted avg       0.97      0.97      0.97       836

Confusion Matrix:
[[708  16]
 [  9 103]]


In [ ]:
# Find incorrectly classified messages
errors = X_test[y_test != y_test_pred]

error_details = pd.DataFrame({
    "Message": errors.values,
    "Actual": y_test[y_test != y_test_pred].values,
    "Predicted": [
        y_test_pred[i]
        for i in range(len(y_test_pred))
        if y_test.iloc[i] != y_test_pred[i]
    ]
})

print(error_details.head(5).to_string(index=False))

                                                                                                                                                       Message Actual Predicted
                                                                                                                           Its normally hot mail. Com you see!    ham      spam
     CLAIRE here am havin borin time & am now alone U wanna cum over 2nite? Chat now 09099725823 hope 2 C U Luv CLAIRE xx Calls£1/minmoremobsEMSPOBox45PO139WA   spam       ham
                                                             We have sent JD for Customer Service cum Accounts Executive to ur mail id, For details contact us    ham      spam
ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 DONT GIV A SHIT. BAILIFF DUE IN DAYS. I O £250 3 WANT £800   spam       ham
                       And stop wondering "wow is she ever going to stop tm'ing me ?!" because I will tm you whenever I 

In [ ]:

local_messages = [
    ("Your JazzCash account has won a cash prize. Claim it now by sharing your PIN.", "spam"),
    ("Apka Easypaisa account band honay wala hai. Verification ke liye OTP share karein.", "spam"),
    ("Congratulations! You have been selected for a job. Pay a registration fee to continue.", "spam"),
    ("Your parcel is waiting. Pay an unexpected delivery fee through this link.", "spam"),
    ("Ammi, main ghar pohanch gaya hoon.", "ham"),
    ("Your electricity bill is due tomorrow. Please check the official app.", "ham"),
    ("I have sent you Rs. 500 through Easypaisa.", "ham"),
    ("Kal school kitne bajay jana hai?", "ham")
]

local_texts = [message for message, label in local_messages]
local_actual = [label for message, label in local_messages]

local_vectors = vectorizer.transform(local_texts)
local_probabilities = model.predict_proba(local_vectors)

spam_index = list(model.classes_).index("spam")
local_scores = local_probabilities[:, spam_index]

local_predictions = [
    "spam" if score >= 0.20 else "ham"
    for score in local_scores
]

for message, actual, predicted, score in zip(
    local_texts, local_actual, local_predictions, local_scores
):
    print("Message:", message)
    print("Expected:", actual)
    print("Predicted:", predicted)
    print("Spam score:", round(float(score) * 100, 2), "%")
    print("-" * 60)


Message: Your JazzCash account has won a cash prize. Claim it now by sharing your PIN.
Expected: spam
Predicted: spam
Spam score: 75.35 %
------------------------------------------------------------
Message: Apka Easypaisa account band honay wala hai. Verification ke liye OTP share karein.
Expected: spam
Predicted: ham
Spam score: 12.61 %
------------------------------------------------------------
Message: Congratulations! You have been selected for a job. Pay a registration fee to continue.
Expected: spam
Predicted: ham
Spam score: 11.33 %
------------------------------------------------------------
Message: Your parcel is waiting. Pay an unexpected delivery fee through this link.
Expected: spam
Predicted: spam
Spam score: 21.97 %
------------------------------------------------------------
Message: Ammi, main ghar pohanch gaya hoon.
Expected: ham
Predicted: ham
Spam score: 8.53 %
------------------------------------------------------------
Message: Your electricity bill is due tomor

In [ ]:

pakistani_training = [
    ("Your JazzCash account is blocked. Share your PIN to unlock it.", "spam"),
    ("Easypaisa verification required. Send your OTP immediately.", "spam"),
    ("Congratulations! Pay a registration fee to secure your job.", "spam"),
    ("Your parcel is held. Pay a fee using this link.", "spam"),
    ("Apka account band ho jayega, OTP bhejein.", "spam"),
    ("Prize jeetne ke liye abhi link par click karein.", "spam"),
    ("Free mobile balance hasil karne ke liye PIN bhejein.", "spam"),
    ("Aapka inaam tayyar hai, claim karne ke liye fee dein.", "spam"),

    ("Ammi, main ghar pohanch gaya hoon.", "ham"),
    ("Kal school kitne bajay jana hai?", "ham"),
    ("I have sent you Rs. 500 through Easypaisa.", "ham"),
    ("Your electricity bill is due tomorrow.", "ham"),
    ("Main 10 minutes mein aa raha hoon.", "ham"),
    ("Aaj homework complete kar lena.", "ham"),
    ("Please call me when you reach home.", "ham"),
    ("Your payment of Rs. 500 was successful.", "ham")
]

pak_df = pd.DataFrame(
    pakistani_training,
    columns=["message", "label"]
)

print(pak_df["label"].value_counts())


label
spam    8
ham     8
Name: count, dtype: int64


In [ ]:

# Combine original training data with Pakistani examples
X_train_combined = pd.concat(
    [X_train, pak_df["message"]],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_train, pak_df["label"]],
    ignore_index=True
)

print("Original training messages:", len(X_train))
print("Pakistani examples added:", len(pak_df))
print("New training size:", len(X_train_combined))
print("\nClass distribution:")
print(y_train_combined.value_counts())


Original training messages: 3900
Pakistani examples added: 16
New training size: 3916

Class distribution:
label
ham     3385
spam     531
Name: count, dtype: int64


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Create a fresh vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

# Convert combined training messages into features
X_train_combined_vectors = vectorizer.fit_transform(
    X_train_combined
)

# Transform validation messages
X_val_vectors = vectorizer.transform(X_val)

# Train the updated model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_combined_vectors, y_train_combined)

print("Model retrained successfully!")
print("Training features:", X_train_combined_vectors.shape)
print("Validation features:", X_val_vectors.shape)


Model retrained successfully!
Training features: (3916, 6900)
Validation features: (836, 6900)


In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score

val_probabilities = model.predict_proba(X_val_vectors)
spam_index = list(model.classes_).index("spam")
spam_scores = val_probabilities[:, spam_index]

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:
    predictions = [
        "spam" if score >= threshold else "ham"
        for score in spam_scores
    ]

    precision = precision_score(
        y_val, predictions, pos_label="spam"
    )
    recall = recall_score(
        y_val, predictions, pos_label="spam"
    )
    f1 = f1_score(
        y_val, predictions, pos_label="spam"
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )


Threshold: 0.10 | Precision: 0.598 | Recall: 0.982 | F1: 0.743
Threshold: 0.15 | Precision: 0.849 | Recall: 0.955 | F1: 0.899
Threshold: 0.20 | Precision: 0.929 | Recall: 0.929 | F1: 0.929
Threshold: 0.25 | Precision: 0.952 | Recall: 0.893 | F1: 0.922
Threshold: 0.30 | Precision: 0.979 | Recall: 0.848 | F1: 0.909
Threshold: 0.35 | Precision: 0.979 | Recall: 0.830 | F1: 0.899
Threshold: 0.40 | Precision: 0.989 | Recall: 0.821 | F1: 0.898
Threshold: 0.50 | Precision: 1.000 | Recall: 0.723 | F1: 0.839


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

THRESHOLD = 0.20

X_test_vectors = vectorizer.transform(X_test)
test_probabilities = model.predict_proba(X_test_vectors)

spam_index = list(model.classes_).index("spam")
spam_scores = test_probabilities[:, spam_index]

y_test_pred = [
    "spam" if score >= THRESHOLD else "ham"
    for score in spam_scores
]

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Accuracy: 0.9665071770334929

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.97      0.98       724
        spam       0.84      0.92      0.88       112

    accuracy                           0.97       836
   macro avg       0.92      0.95      0.93       836
weighted avg       0.97      0.97      0.97       836


Confusion Matrix:
[[705  19]
 [  9 103]]


In [ ]:

pakistani_test = [
    ("JazzCash: Your wallet is suspended. Send your OTP to restore access.", "spam"),
    ("Easypaisa users must pay a verification charge to receive a reward.", "spam"),
    ("You have been shortlisted. Transfer a processing fee to confirm your job.", "spam"),
    ("Your mobile number has won a lucky draw. Claim your prize now.", "spam"),
    ("Abu, main school se ghar aa gaya hoon.", "ham"),
    ("Please send me the electricity bill when you get home.", "ham"),
    ("I received your payment. Thank you!", "ham"),
    ("Kal hum 5 bajay milain ge.", "ham")
]

pak_texts = [m for m, label in pakistani_test]
pak_actual = [label for m, label in pakistani_test]

pak_vectors = vectorizer.transform(pak_texts)
pak_scores = model.predict_proba(pak_vectors)[:, spam_index]

pak_predictions = [
    "spam" if score >= 0.20 else "ham"
    for score in pak_scores
]

for message, actual, predicted, score in zip(
    pak_texts, pak_actual, pak_predictions, pak_scores
):
    print("Message:", message)
    print("Actual:", actual)
    print("Predicted:", predicted)
    print("Spam score:", round(float(score) * 100, 2), "%")
    print("-" * 50)


Message: JazzCash: Your wallet is suspended. Send your OTP to restore access.
Actual: spam
Predicted: spam
Spam score: 22.45 %
--------------------------------------------------
Message: Easypaisa users must pay a verification charge to receive a reward.
Actual: spam
Predicted: spam
Spam score: 26.51 %
--------------------------------------------------
Message: You have been shortlisted. Transfer a processing fee to confirm your job.
Actual: spam
Predicted: ham
Spam score: 10.73 %
--------------------------------------------------
Message: Your mobile number has won a lucky draw. Claim your prize now.
Actual: spam
Predicted: spam
Spam score: 91.52 %
--------------------------------------------------
Message: Abu, main school se ghar aa gaya hoon.
Actual: ham
Predicted: ham
Spam score: 7.14 %
--------------------------------------------------
Message: Please send me the electricity bill when you get home.
Actual: ham
Predicted: ham
Spam score: 6.7 %
-------------------------------------

In [ ]:

def explain_prediction(message, top_n=5):
    message_vector = vectorizer.transform([message])

    feature_names = vectorizer.get_feature_names_out()

    # Find the words present in the message
    nonzero_indices = message_vector.nonzero()[1]

    contributions = []

    # Confirm that the model's spam class is the positive class
    spam_index = list(model.classes_).index("spam")

    # Logistic Regression coefficients point towards classes_[1]
    direction = 1 if spam_index == 1 else -1

    for index in nonzero_indices:
        word = feature_names[index]

        contribution = (
            message_vector[0, index]
            * model.coef_[0][index]
            * direction
        )

        if contribution > 0:
            contributions.append((word, float(contribution)))

    contributions.sort(key=lambda x: x[1], reverse=True)

    if not contributions:
        return "No strong spam-related word signals were identified."

    return "\n".join(
        [f"- {word}" for word, score in contributions[:top_n]]
    )


# Test the explanation
test_message = "Congratulations! You have won a cash prize. Claim it now."

print("Message:", test_message)
print("Words influencing spam prediction:")
print(explain_prediction(test_message))


Message: Congratulations! You have won a cash prize. Claim it now.
Words influencing spam prediction:
- claim
- prize
- won
- cash
- congratulations


In [ ]:
import joblib

# Save the current Pakistani-enhanced experiment separately
joblib.dump(model, "scamshield_pakistani_experiment.pkl")
joblib.dump(vectorizer, "scamshield_pakistani_vectorizer.pkl")

print("Pakistani experiment saved successfully!")

Pakistani experiment saved successfully!


In [ ]:

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Restore the original English training setup
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_vectors = vectorizer.fit_transform(X_train)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectors, y_train)

# Save the original English model separately
joblib.dump(model, "scamshield_model.pkl")
joblib.dump(vectorizer, "scamshield_vectorizer.pkl")

print("Original English model restored and saved!")


Original English model restored and saved!


In [ ]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

THRESHOLD = 0.20

X_test_vectors = vectorizer.transform(X_test)
test_probabilities = model.predict_proba(X_test_vectors)

spam_index = list(model.classes_).index("spam")
spam_scores = test_probabilities[:, spam_index]

y_test_pred = [
    "spam" if score >= THRESHOLD else "ham"
    for score in spam_scores
]

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Accuracy: 0.9700956937799043

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       724
        spam       0.87      0.92      0.89       112

    accuracy                           0.97       836
   macro avg       0.93      0.95      0.94       836
weighted avg       0.97      0.97      0.97       836


Confusion Matrix:
[[708  16]
 [  9 103]]


In [ ]:
for actual, predicted, message in zip(y_test, y_test_pred, X_test):
    if actual != predicted:
        print("Message:", message)
        print("Actual:", actual)
        print("Predicted:", predicted)
        print("-" * 60)

Message: Its normally hot mail. Com you see!
Actual: ham
Predicted: spam
------------------------------------------------------------
Message: CLAIRE here am havin borin time & am now alone U wanna cum over 2nite? Chat now 09099725823 hope 2 C U Luv CLAIRE xx Calls£1/minmoremobsEMSPOBox45PO139WA
Actual: spam
Predicted: ham
------------------------------------------------------------
Message: We have sent JD for Customer Service cum Accounts Executive to ur mail id, For details contact us
Actual: ham
Predicted: spam
------------------------------------------------------------
Message: ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE MINS. INDIA CUST SERVs SED YES. L8ER GOT MEGA BILL. 3 DONT GIV A SHIT. BAILIFF DUE IN DAYS. I O £250 3 WANT £800
Actual: spam
Predicted: ham
------------------------------------------------------------
Message: And stop wondering "wow is she ever going to stop tm'ing me ?!" because I will tm you whenever I want because you are MINE ... *laughs*
Actual: ham
Pred

In [ ]:
import pandas as pd

errors = pd.DataFrame({
    "Message": list(X_test),
    "Actual": list(y_test),
    "Predicted": list(y_test_pred)
})

false_positives = errors[
    (errors["Actual"] == "ham") &
    (errors["Predicted"] == "spam")
].head(2)

false_negatives = errors[
    (errors["Actual"] == "spam") &
    (errors["Predicted"] == "ham")
].head(3)

five_errors = pd.concat(
    [false_positives, false_negatives]
)

display(five_errors)

,Message,Actual,Predicted
92,Its normally hot mail. Com you see!,ham,spam
183,We have sent JD for Customer Service cum Accou...,ham,spam
95,CLAIRE here am havin borin time & am now alone...,spam,ham
248,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...,spam,ham
285,"Latest News! Police station toilet stolen, cop...",spam,ham


In [ ]:

import pandas as pd
import joblib
from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Additional Pakistani-context training examples
extra_examples = [
    # Suspicious job offers
    ("You have been selected for a job. Pay a processing fee to confirm your position.", "spam"),
    ("Congratulations! Your job application is approved. Pay a registration fee now.", "spam"),
    ("Work from home and earn Rs. 5000 daily. Pay an activation fee to start.", "spam"),
    ("Your interview is confirmed. Transfer the required fee to secure your job.", "spam"),
    ("Limited vacancies available. Send money to guarantee your employment.", "spam"),
    ("You won a job opportunity. Click this link and pay the verification fee.", "spam"),
    ("Pay Rs. 1500 to process your job application immediately.", "spam"),
    ("Your job offer will expire today. Send the fee to claim it.", "spam"),

    # Legitimate messages
    ("Your monthly bank statement is ready. View it through the official mobile app.", "ham"),
    ("Your bank statement for October is available in your online banking account.", "ham"),
    ("Your account balance has been updated after your recent transaction.", "ham"),
    ("Your payment was received successfully. Thank you for banking with us.", "ham"),
    ("Your electricity bill is due tomorrow. Please use the official app to check it.", "ham"),
    ("Your salary has been credited to your bank account.", "ham"),
    ("Your transaction confirmation is available in your mobile banking application.", "ham"),
    ("Your monthly account summary is ready to view through online banking.", "ham")
]

extra_df = pd.DataFrame(extra_examples, columns=["message", "label"])

# Add examples ONLY to the training data
X_train_combined = pd.concat(
    [X_train.reset_index(drop=True), extra_df["message"]],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_train.reset_index(drop=True), extra_df["label"]],
    ignore_index=True
)

# Retrain vectorizer and model
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_vectors = vectorizer.fit_transform(X_train_combined)
X_val_vectors = vectorizer.transform(X_val)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectors, y_train_combined)

# Evaluate on the untouched validation and test sets
THRESHOLD = 0.20

X_test_vectors = vectorizer.transform(X_test)
test_probabilities = model.predict_proba(X_test_vectors)

spam_index = list(model.classes_).index("spam")
spam_scores = test_probabilities[:, spam_index]

y_test_pred = [
    "spam" if score >= THRESHOLD else "ham"
    for score in spam_scores
]

print("UPDATED MODEL TRAINED")
print("Original training examples:", len(X_train))
print("Additional examples:", len(extra_df))
print("Total training examples:", len(X_train_combined))

print("\nUPDATED TEST ACCURACY:")
print(accuracy_score(y_test, y_test_pred))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_test_pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_test_pred))

# Save updated files
joblib.dump(model, "scamshield_model.pkl")
joblib.dump(vectorizer, "scamshield_vectorizer.pkl")

print("\nUpdated model and vectorizer saved!")

# Download only the two updated model files
files.download("scamshield_model.pkl")
files.download("scamshield_vectorizer.pkl")


UPDATED MODEL TRAINED
Original training examples: 3900
Additional examples: 16
Total training examples: 3916

UPDATED TEST ACCURACY:
0.9677033492822966

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       724
        spam       0.85      0.92      0.88       112

    accuracy                           0.97       836
   macro avg       0.92      0.95      0.93       836
weighted avg       0.97      0.97      0.97       836


CONFUSION MATRIX:
[[706  18]
 [  9 103]]

Updated model and vectorizer saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>